# Quadrant data creation process

In [ ]:
from pathlib import Path
from typing import Optional

import numpy as np

from automated_underwater_area_estimation.segmentation_quadrant.model import (
    QuadrantSegmentationModel,
)
from PIL import Image
import json
import matplotlib.pyplot as plt
def load_image_rgb(path: str | Path) -> np.ndarray:
    """RGB (H,W,3) uint8."""
    return np.asarray(Image.open(path).convert("RGB"))

def load_mask(path: str | Path) -> np.ndarray:
    """
    Binary mask loader for image files or .pt (dict['mask'] or tensor).
    Returns uint8 {0,1}.
    """
    p = str(path)
    if p.lower().endswith(".pt"):
        import torch
        t = torch.load(p, map_location="cpu")
        if isinstance(t, dict) and "mask" in t:
            t = t["mask"]
        if t.ndim == 3 and t.shape[0] in (1, 3):
            t = t.max(dim=0).values
        m = (t > 0.5).to(torch.uint8).cpu().numpy()
        return m
    m = np.asarray(Image.open(p).convert("L"))
    return (m > 127).astype(np.uint8)

def resize_mask_to_image(mask01: np.ndarray, image_rgb: np.ndarray) -> np.ndarray:
    """Nearest-neighbor resize of mask to image size (H,W)."""
    h, w = image_rgb.shape[:2]
    if mask01.shape[:2] == (h, w):
        return mask01
    pil = Image.fromarray((mask01 * 255).astype(np.uint8))
    pil = pil.resize((w, h), resample=Image.NEAREST)
    return (np.asarray(pil) > 127).astype(np.uint8)

def read_clicks_from_json(clicks_json_path: str | Path) -> tuple[np.ndarray, np.ndarray, tuple[int,int] | None]:
    """
    Returns (pos Nx2, neg Nx2, save_size (W,H) or None) in ORIGINAL image coordinates.
    Uses 'clicks_original': each item = [x,y,label] with label 1=pos, 0=neg.
    """
    data = json.loads(Path(clicks_json_path).read_text(encoding="utf-8"))
    pts = np.asarray(data.get("clicks_original", []), dtype=float)
    pos = pts[pts[:, 2] == 1][:, :2] if pts.size else np.empty((0, 2))
    neg = pts[pts[:, 2] == 0][:, :2] if pts.size else np.empty((0, 2))
    save_size = tuple(data["save_size"]) if "save_size" in data else None  # [W,H]
    return pos, neg, save_size

def scale_points_to_image(pts: np.ndarray, src_size_wh: tuple[int,int] | None, img_w: int, img_h: int) -> np.ndarray:
    if pts.size == 0 or src_size_wh is None:
        return pts
    sx = img_w / float(src_size_wh[0])
    sy = img_h / float(src_size_wh[1])
    out = pts.copy()
    out[:, 0] *= sx
    out[:, 1] *= sy
    return out

def overlay_mask_on_image(image_rgb: np.ndarray,
                          mask01: np.ndarray,
                          color=(255, 0, 255),
                          alpha: float = 0.35) -> np.ndarray:
    """
    Alpha-blend a 1-channel binary mask onto an RGB image with NumPy broadcasting.
    image_rgb: (H,W,3) uint8
    mask01   : (H,W)   {0,1} or bool
    """
    img = image_rgb.astype(np.float32)
    # color layer
    overlay = np.zeros_like(img, dtype=np.float32)
    overlay[..., 0], overlay[..., 1], overlay[..., 2] = color

    # broadcast mask to (H,W,1) as float (0 or 1)
    m = mask01.astype(np.float32)[..., None]

    # blend only where mask==1; broadcasting handles the 3 channels
    out = img * (1.0 - alpha * m) + overlay * (alpha * m)
    return np.clip(out, 0, 255).astype(np.uint8)


# ---------------------------
# Main: 2×2 visualization
# ---------------------------
def visualize_four_panel(
    image_path: str | Path,
    clicks_json_path: str | Path,
    gt_mask_path: str | Path,
    improved_mask_path: str | Path,
    save_path: str | Path,
    model: Optional[QuadrantSegmentationModel] = None,
    point_size: int = 60,
) -> str:
    """
    Panels:
      [0,0] Original + clicks
      [0,1] GT mask overlay + clicks
      [1,0] Improved mask (grayscale)
      [1,1] Model prediction overlay
    """
    # Image
    image = load_image_rgb(image_path)
    H, W = image.shape[:2]

    # Clicks (original coordinates), scale if JSON save_size doesn't match actual image size
    pos, neg, save_size = read_clicks_from_json(clicks_json_path)
    pos = scale_points_to_image(pos, save_size, W, H)
    neg = scale_points_to_image(neg, save_size, W, H)

    # Masks
    gt_mask   = resize_mask_to_image(load_mask(gt_mask_path), image)
    imp_mask  = resize_mask_to_image(load_mask(improved_mask_path), image)

    # Model prediction
    if model is None:
        model = QuadrantSegmentationModel()
    pil_img = Image.fromarray(image)
    pred_mask = model.segment_image(pil_img)          # expect HxW {0,1}/bool
    if isinstance(pred_mask, np.ndarray):
        pass
    else:
        # torch/tensor -> numpy
        try:
            pred_mask = pred_mask.detach().cpu().numpy()
        except Exception:
            pred_mask = np.asarray(pred_mask)
    if pred_mask.dtype != np.uint8:
        pred_mask = (pred_mask > 0.5).astype(np.uint8)
    pred_mask = resize_mask_to_image(pred_mask, image)

    # Compose panels
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # (1) original + clicks
    ax = axes[0, 0]
    ax.imshow(image)
    if neg.size: ax.scatter(neg[:, 0], neg[:, 1], s=point_size, marker="x", label="neg", color="red")
    if pos.size: ax.scatter(pos[:, 0], pos[:, 1], s=point_size, marker="o", label="pos", color="green")
    ax.set_title("Original image with clicks", fontsize=28)
    ax.axis("off")

    # (2) GT overlay + clicks
    ax = axes[0, 1]
    gt_overlay = overlay_mask_on_image(image, gt_mask, color=(217,95,2), alpha=0.4)
    ax.imshow(gt_overlay)
    if neg.size: ax.scatter(neg[:, 0], neg[:, 1], s=point_size, marker="x", color="red")
    if pos.size: ax.scatter(pos[:, 0], pos[:, 1], s=point_size, marker="o", color="green")
    ax.set_title("Generated mask", fontsize=28)
    ax.axis("off")

    # (3) improved mask (standalone)
    ax = axes[1, 0]
    pred_overlay = overlay_mask_on_image(image, imp_mask, color=(217,95,2), alpha=0.4)
    ax.imshow(pred_overlay)
    ax.set_title("Improved mask", fontsize=28)
    ax.axis("off")
    # ax.imshow(imp_mask, cmap="gray")
    # ax.set_title("Improved mask", fontsize=28)
    # ax.axis("off")

    # (4) model prediction overlay
    ax = axes[1, 1]
    pred_overlay = overlay_mask_on_image(image, pred_mask, color=(255,0,255), alpha=0.4)
    ax.imshow(pred_overlay)
    ax.set_title("Our Model prediction", fontsize=28)
    ax.axis("off")

    plt.tight_layout()
    return fig

In [ ]:
base = Path("./automated_underwater_area_estimation/data_preprocessed/IBF")
name = "GA 1_PB100637"
img_path = base / f"images/{name}.JPG"
clicks   = base / f"clicks/{name}_metadata.json"
gt_mask  = base / f"masks/{name}.pt"
imp_mask = base / f"improved_masks/{name}_improved.pt"
out_path = base / "viz_4panel.png"

output = visualize_four_panel(img_path, clicks, gt_mask, imp_mask, out_path)

In [ ]:
# from pathlib import Path
# from typing import Optional, List
#
# import numpy as np
# import matplotlib.pyplot as plt
# from PIL import Image
#
# from automated_underwater_area_estimation.segmentation_quadrant.model import (
#     QuadrantSegmentationModel,
# )
#
# def _save_fig(img: np.ndarray, out_path: Path, dpi: int = 300) -> Path:
#     """Save an RGB array as a figure with no axes/frames/titles."""
#     # size roughly matches pixels (not critical since we set bbox/pad)
#     fig, ax = plt.subplots(figsize=(img.shape[1] / 100, img.shape[0] / 100), dpi=100)
#     ax.imshow(img)
#     ax.axis("off")
#     plt.subplots_adjust(left=0, right=1, bottom=0, top=1)
#     plt.close(fig)
#     return out_path
#
# def visualize_panels_as_list(
#     image_path: str | Path,
#     clicks_json_path: str | Path,
#     gt_mask_path: str | Path,
#     improved_mask_path: str | Path,
#     save_dir: str | Path,
#     *,
#     model: Optional[QuadrantSegmentationModel] = None,
#     point_size: int = 60,
#     dpi: int = 300,
# ) -> List[Path]:
#     """
#     Order preserved:
#       1. Original + clicks
#       2. GT overlay + clicks
#       3. Improved mask overlay
#       4. Model prediction overlay
#     Returns a list of saved paths in that order.
#     """
#
#     # --- Load image and clicks ---
#     image = load_image_rgb(image_path)
#     H, W = image.shape[:2]
#
#     pos, neg, save_size = read_clicks_from_json(clicks_json_path)
#     pos = scale_points_to_image(pos, save_size, W, H)
#     neg = scale_points_to_image(neg, save_size, W, H)
#
#     # --- Masks ---
#     gt_mask  = resize_mask_to_image(load_mask(gt_mask_path), image)
#     imp_mask = resize_mask_to_image(load_mask(improved_mask_path), image)
#
#     # --- Model prediction ---
#     if model is None:
#         model = QuadrantSegmentationModel()
#     pred = model.segment_image(Image.fromarray(image))
#     if not isinstance(pred, np.ndarray):
#         try:
#             pred = pred.detach().cpu().numpy()
#         except Exception:
#             pred = np.asarray(pred)
#     pred_mask = (pred > 0.5).astype(np.uint8) if pred.dtype != np.uint8 else pred
#     pred_mask = resize_mask_to_image(pred_mask, image)
#
#     outputs: List[Path] = []
#
#     # 1) Original + clicks
#     fig, ax = plt.subplots()
#     ax.imshow(image)
#     if neg.size: ax.scatter(neg[:, 0], neg[:, 1], s=point_size, marker="x", color="red")
#     if pos.size: ax.scatter(pos[:, 0], pos[:, 1], s=point_size, marker="o", color="green")
#     ax.axis("off")
#     plt.subplots_adjust(left=0, right=1, bottom=0, top=1)
#     out1 = save_dir / "panel_1_original_with_clicks.png"
#     plt.close(fig)
#     outputs.append(out1)
#
#     # 2) GT overlay + clicks
#     gt_overlay = overlay_mask_on_image(image, gt_mask, color=(217, 95, 2), alpha=0.4)
#     fig, ax = plt.subplots()
#     ax.imshow(gt_overlay)
#     if neg.size: ax.scatter(neg[:, 0], neg[:, 1], s=point_size, marker="x", color="red")
#     if pos.size: ax.scatter(pos[:, 0], pos[:, 1], s=point_size, marker="o", color="green")
#     ax.axis("off")
#     plt.subplots_adjust(left=0, right=1, bottom=0, top=1)
#     out2 = save_dir / "panel_2_gt_overlay_with_clicks.png"
#     plt.close(fig)
#     outputs.append(out2)
#
#     # 3) Improved mask overlay
#     imp_overlay = overlay_mask_on_image(image, imp_mask, color=(217, 95, 2), alpha=0.4)
#     out3 = _save_fig(imp_overlay, save_dir / "panel_3_improved_mask_overlay.png", dpi)
#     outputs.append(out3)
#
#     # 4) Model prediction overlay
#     pred_overlay = overlay_mask_on_image(image, pred_mask, color=(255, 0, 255), alpha=0.4)
#     out4 = _save_fig(pred_overlay, save_dir / "panel_4_model_prediction_overlay.png", dpi)
#     outputs.append(out4)
#
#     return outputs


In [ ]:
# output = visualize_panels_as_list(img_path, clicks, gt_mask, imp_mask, out_path)

# Images pipelines

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from pathlib import Path

baseline = "GA 1_PB100637"
image_path = Path(f"./automated_underwater_area_estimation/data_preprocessed/IBF/images/{baseline}.JPG")
figsize = (8, 6)
titlefont = 20
alpha = 0.6

# Load image
img = Image.open(image_path).convert("RGB")
img_np = np.array(img)

# Plot
fig, ax = plt.subplots(figsize=figsize)
ax.imshow(img_np)
ax.set_title("Input image", fontsize=titlefont)
ax.axis('off')  # hide axes

# Remove margins:
ax.set_position([0, 0, 1, 1])     # Fill entire figure with the axes
plt.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0, hspace=0)

plt.tight_layout(pad=0)            # Remove padding
plt.margins(0, 0)                  # Remove data margins
plt.show()


In [ ]:
from automated_underwater_area_estimation.segmentation_corals.epfl.model import EPFLModel
coral_segmentation_model = EPFLModel("EPFL-ECEO/segformer-b5-finetuned-coralscapes-1024-1024")
img_prep, coral_segmentation_mask = coral_segmentation_model.segment_image(img, adjust_size=False, use_sliding_window=True)
mask = coral_segmentation_mask.cpu()
img_arr = np.array(img.convert("RGB"))

# Create an overlay image by blending: choose a colour for the mask (e.g., red) and alpha blending
overlay = img_arr.copy()
color = (255, 0, 0)
colour = np.array(color, dtype=np.uint8)     # red colour for segmentation

# Apply colour to overlay where mask is true
overlay[mask] = (overlay[mask] * (1 - alpha) + colour * alpha).astype(np.uint8)

# Plot it
plt.figure(figsize=figsize)
plt.imshow(overlay)
plt.title("Segment corals", fontsize=titlefont)   # you can increase fontsize if needed
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()

In [ ]:
from automated_underwater_area_estimation.segmentation_quadrant.model import QuadrantSegmentationModel

quad_model = QuadrantSegmentationModel()
mask_quad = quad_model.segment_image(img)
mask_quad = mask_quad.astype(bool)
overlay_quad = img_arr.copy()
color = (0, 0, 255)
colour_blue = np.array(color, dtype=np.uint8)
overlay_quad[mask_quad] = (overlay_quad[mask_quad] * (1-alpha) + colour_blue * alpha).astype(np.uint8)

plt.figure(figsize=figsize)
plt.imshow(overlay_quad)
plt.title("Segment quadrant", fontsize=titlefont)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Union, Dict, Tuple

def to_mask2d(m_or_path: Union[str, np.ndarray]) -> np.ndarray:
    m = np.asarray(m_or_path)
    m = (np.squeeze(m) > 0).astype(np.uint8)
    if m.ndim != 2:
        raise ValueError(f"Expected 2D mask, got shape {m.shape}")
    return m

def point_bottom_right(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    if ys.size == 0: raise ValueError("Mask is empty.")
    i = (ys.astype(np.int64) * xs.astype(np.int64)).argmax()
    return int(ys[i]), int(xs[i])

def point_top_left(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    keep = (ys > 0) & (xs > 0)
    ys, xs = ys[keep], xs[keep]
    if ys.size == 0: raise ValueError("No white pixels with x>0 and y>0.")
    sums = ys.astype(np.int64) + xs.astype(np.int64)
    min_sum = sums.min()
    idx = np.where(sums == min_sum)[0]
    y_sel = int(ys[idx].min())
    x_sel = int(xs[idx][ys[idx] == y_sel].min())
    return y_sel, x_sel

def point_top_right(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    diffs = xs.astype(np.int64) - ys.astype(np.int64)
    best = diffs.max()
    keep = (diffs == best)
    xs_t, ys_t = xs[keep], ys[keep]
    x_sel = int(xs_t.max())
    y_sel = int(ys_t[xs_t == x_sel].min())
    return y_sel, x_sel

def point_bottom_left(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    diffs = ys.astype(np.int64) - xs.astype(np.int64)
    best = diffs.max()
    keep = (diffs == best)
    ys_t, xs_t = ys[keep], xs[keep]
    y_sel = int(ys_t.max())
    x_sel = int(xs_t[ys_t == y_sel].min())
    return y_sel, x_sel

def find_four_points(mask_2d: Union[str, np.ndarray]) -> Dict[str, Tuple[int,int]]:
    m = to_mask2d(mask_2d)
    return {
        "TL": point_top_left(m),
        "TR": point_top_right(m),
        "BR": point_bottom_right(m),
        "BL": point_bottom_left(m),
    }

pt_color = {"TL": "#FFBF00", "TR": "#FFBF00", "BR": "#FFBF00", "BL": "#FFBF00"}
side_color = "#FFBF00"
diag_colors = {"TL_BR": "#FFBF00", "TR_BL": "#FFBF00"}

pts = find_four_points(mask_quad)

fig, ax = plt.subplots(figsize=figsize)
ax.imshow(overlay_quad)  # overlay from your code above

# plot corner points + labels
for k, (y, x) in pts.items():
    ax.scatter([x], [y], s=150, facecolor="white", edgecolor="black", zorder=4)
    ax.scatter([x], [y], s=70, color=pt_color[k], zorder=5)

# draw sides and diagonals
sides = [("TR", "TL"), ("TR", "BR"), ("BL", "BR"), ("TL", "BL")]
diags = [("TL", "BR"), ("TR", "BL")]

def draw_seg(a, b, color, lw=2.4, z=2):
    (y1, x1), (y2, x2) = pts[a], pts[b]
    ax.plot([x1, x2], [y1, y2], color=color, lw=lw, alpha=0.9, zorder=z)

for a, b in sides:
    draw_seg(a, b, side_color, lw=2.2, z=2)
for (a, b), key in zip(diags, ["TL_BR", "TR_BL"]):
    draw_seg(a, b, diag_colors[key], lw=2.4, z=1)

ax.set_title("Quadrant identification", fontsize=titlefont)
ax.set_axis_off()
plt.tight_layout(pad=0)
plt.show()

# Check

In [ ]:
from pathlib import Path
import json
import pandas as pd

def load_all_comparisons(root_dir: str) -> pd.DataFrame:
    root = Path(root_dir)
    rows = []
    for fp in root.rglob("comparison/model_comparison.json"):
        try:
            data = json.loads(fp.read_text())
            dataset_name = fp.parent.parent.name
            for rec in data:
                rec = dict(rec)  # copy
                rec.setdefault("dataset", dataset_name)
                rec["source_file"] = str(fp)
                rows.append(rec)
        except Exception as e:
            print(f"⚠️ Skipping {fp}: {e}")

    df = pd.DataFrame(rows)

    # Nice column order if present
    pref_order = [
        "dataset", "model",
        "dice_mean", "dice_std",
        "iou_mean", "iou_std",
        "precision_mean", "precision_std",
        "recall_mean", "recall_std",
        "pixel_accuracy_mean", "pixel_accuracy_std",
        "source_file",
    ]
    cols = [c for c in pref_order if c in df.columns] + [c for c in df.columns if c not in pref_order]
    return df[cols]

# ---- use it ----
df_all = load_all_comparisons("./automated_underwater_area_estimation/evaluation_results")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==== CONFIG ====
dataset_filter = None  # e.g., "UNAL_BLEACHING_TAYRONA" or None
series = [
    ("dice_mean", "dice_std", "DICE Similarity Coefficient (DSC)"),
    ("iou_mean",  "iou_std",  "Intersection over Union (IoU)"),
]
colors = ["#66c2a5", "#fc8d62"]  # colorblind-friendly

# ==== PREP (expects df_all from your earlier step) ====
dfp = df_all.copy()
dfp["model"] = dfp["model"].str.replace("_yolov8", "").str.replace("_latest", "")
if dataset_filter:
    dfp = dfp[dfp["dataset"] == dataset_filter]

# If multiple rows per model, average them
metrics_means = [m for m, _, _ in series]
metrics_stds  = [s for _, s, _ in series]
use_cols = ["model"] + metrics_means + metrics_stds
dfp = dfp[use_cols].groupby("model", as_index=False).mean().sort_values(by=metrics_means[0], ascending=False)

models = dfp["model"].tolist()
x = np.arange(len(models))
n_series = len(series)
group_w = 0.8
bar_w = group_w / n_series

# ==== PLOT ====
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(15, 6), dpi=300)

for i, (m_mean, m_std, label) in enumerate(series):
    ax.bar(
        x - group_w/2 + i*bar_w + bar_w/2,
        dfp[m_mean].values,
        yerr=dfp[m_std].values if m_std in dfp.columns else None,
        width=bar_w,
        label=label,
        edgecolor="white",
        linewidth=0.7,
        color=colors[i % len(colors)],
        alpha=0.95,
        capsize=4,
    )

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=18, fontweight="bold")
ax.tick_params(axis='y', labelsize=18)
ax.set_ylim(0, 1)
ax.set_ylabel("Score", fontsize=18, fontweight="bold")
ax.set_xlabel("Models", fontsize=18, fontweight="bold")
ax.set_title(f"Coral Segmentation Evaluation by Model", fontsize=28, fontweight="bold")
ax.legend(title="Metrics", frameon=True, framealpha=0.9, fontsize=18, title_fontsize=18, ncol=1)
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re
from itertools import cycle

# ==== CONFIG ====
dataset_filter = None          # e.g., "UNAL_BLEACHING_TAYRONA" or None
IMG_FMT = "png"                # png / pdf / svg
DPI = 300

# If your column names differ, add aliases here (first match wins):
ALIASES = {
    "dice_mean":              ["dice_mean", "dsc_mean"],
    "dice_std":               ["dice_std", "dsc_std"],
    "iou_mean":               ["iou_mean", "jaccard_mean"],
    "iou_std":                ["iou_std", "jaccard_std"],
    "recall_mean":            ["recall_mean", "tpr_mean", "sensitivity_mean"],
    "recall_std":             ["recall_std", "tpr_std", "sensitivity_std"],
    "precision_mean":         ["precision_mean", "ppv_mean"],
    "precision_std":          ["precision_std", "ppv_std"],
    "pixel_accuracy_mean":    ["pixel_accuracy_mean", "pa_mean", "accuracy_mean"],
    "pixel_accuracy_std":     ["pixel_accuracy_std", "pa_std", "accuracy_std"],
}

# Four metrics to show in ONE figure (2x2 grid)
METRICS = [
    ("dice_mean",       "dice_std",       "Dice score"),
    ("iou_mean",        "iou_std",        "Intersection over Union (IoU)"),
    ("recall_mean",     "recall_std",     "Recall"),
    ("precision_mean",  "precision_std",  "Precision"),
]

# Colorblind-friendly palette (Tol/IBM-style); will cycle if models > len(colors)
MODEL_COLORS = [
    "#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2",
    "#B279A2", "#FF9DA6", "#9D755D", "#BAB0AC"
]

# ==== HELPERS ====
def resolve_col(df: pd.DataFrame, key: str) -> str | None:
    for cand in ALIASES.get(key, [key]):
        if cand in df.columns:
            return cand
    return None

def need_cols(df: pd.DataFrame, keys):
    miss = [k for k in keys if resolve_col(df, k) is None]
    if miss:
        raise KeyError(f"Missing expected metric columns (or aliases): {miss}")

# ==== PREP (expects df_all from your earlier step) ====
dfp = df_all.copy()
dfp["model"] = (
    dfp["model"]
    .str.replace("_yolov8", "", regex=False)
    .str.replace("_latest", "", regex=False)
)

if dataset_filter:
    dfp = dfp[dfp["dataset"] == dataset_filter]

# Average duplicates per model
dfp = dfp.groupby("model", as_index=False).mean(numeric_only=True)

# Ensure required columns exist
need_cols(dfp, [m for m, _, _ in METRICS])

# Fix a SINGLE consistent model order using the first metric (e.g., Dice)
base_mean_col = resolve_col(dfp, METRICS[0][0])
dfp = dfp.sort_values(base_mean_col, ascending=False).reset_index(drop=True)

models = dfp["model"].tolist()

# Assign a consistent color per model across ALL subplots
color_cycle = cycle(MODEL_COLORS)
model_to_color = {m: c for m, c in zip(models, color_cycle)}

# ==== ONE FIGURE with 4 SUBPLOTS ====
plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 9), dpi=DPI)
axes = axes.ravel()

for ax, (m_mean, m_std, title) in zip(axes, METRICS):
    mean_col = resolve_col(dfp, m_mean)
    std_col  = resolve_col(dfp, m_std) if resolve_col(dfp, m_std) else None

    # Keep the same model order for all subplots
    d = dfp[["model", mean_col] + ([std_col] if std_col else [])]

    x = np.arange(len(d))
    heights = d[mean_col].values
    yerr = d[std_col].values if std_col is not None else None

    # Bar colors per model (consistent across all subplots)
    bar_colors = [model_to_color[m] for m in d["model"]]

    ax.bar(
        x, heights,
        yerr=yerr,
        color=bar_colors,
        edgecolor="white",
        linewidth=0.7,
        alpha=0.95,
        capsize=4 if yerr is not None else 0,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(d["model"].tolist(), rotation=20, ha="right", fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score", fontsize=12, fontweight="bold")
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.3)

# Build a single legend mapping colors → models (placed outside)
handles = [plt.Line2D([0], [0], color=model_to_color[m], lw=12) for m in models]
fig.legend(
    handles, models, title="Models", ncol=min(5, len(models)),
    loc="upper center", bbox_to_anchor=(0.5, 1.02),
    frameon=True, framealpha=0.9, fontsize=11, title_fontsize=14
)

fig.suptitle("   ", fontsize=16, fontweight="bold", y=1.06)
fig.tight_layout(rect=[0, 0, 1, 0.98])

# Save if needed
# out_name = f"metrics_2x2_{'all' if dataset_filter is None else dataset_filter}.{IMG_FMT}"
# fig.savefig(out_name, dpi=DPI, bbox_inches="tight")

plt.show()
plt.close(fig)


In [ ]:
df = pd.read_json("./automated_underwater_area_estimation/segmentation_quadrant/training_logs.jsonl", lines=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_training_metrics(df, out_png="metrics.png", out_svg="metrics.svg"):
    """
    Expects columns:
      epoch, eval_miou, eval_dice, eval_boundary_f1
    """
    # --- figure + style ---
    plt.rcParams.update({
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "font.size": 12,
        "axes.labelsize": 13,
        "axes.titlesize": 14,
        "legend.fontsize": 11,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.grid": True,
        "grid.alpha": 0.3,
    })

    metrics = [
        ("eval_miou",        "Intersection over Union (IoU)"),
        ("eval_dice",        "Dice Score"),
        ("eval_boundary_f1", "Boundary F1"),
    ]
    colors      = ["#4C78A8", "#54A24B", "#E45756"]
    markers     = ["o", "s", "^"]

    fig, ax = plt.subplots(figsize=(7.0, 4.2))

    x = df["epoch"].to_numpy()
    lines = []
    labels = []

    for (col, label), c, mk in zip(metrics, colors, markers):
        if col not in df.columns:
            continue
        y = df[col].to_numpy(dtype=float)

        # plot line
        (ln,) = ax.plot(
            x, y, lw=2.0, color=c, marker=mk, markevery=max(len(x)//12, 1),
            mec="white", mew=0.8, ms=5, alpha=0.95
        )
        lines.append(ln)
        labels.append(label)

    # axes formatting
    buffer = 1
    ax.set_xlim(x.min()-buffer, x.max()+buffer)
    ax.set_ylim(0.875, 1.0)
    # ax.set_ylim(0, 1.0)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    # ax.set_title("Validation Metrics over Training")

    # minor ticks/grid for readability
    ax.minorticks_on()
    ax.grid(which="major", axis="both", linestyle="--", alpha=0.30)
    ax.grid(which="minor", axis="y", linestyle=":", alpha=0.20)

    # legend outside to avoid covering data
    ax.legend(lines, labels, title="Metric", frameon=True, framealpha=0.9,
              loc="lower right",)

    fig.tight_layout()
    plt.show()

# Usage:
plot_training_metrics(df)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_training_metrics(df):
    """
    Expects columns:
      epoch, eval_miou, eval_dice, eval_boundary_f1
    """
    # --- figure + style ---
    plt.rcParams.update({
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "font.size": 12,
        "axes.labelsize": 13,
        "axes.titlesize": 14,
        "legend.fontsize": 11,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.grid": True,
        "grid.alpha": 0.3,
    })

    metrics = [
        ("eval_loss", "Loss on Evaluation Set"),
    ]
    colors      = ["#4C78A8"]
    markers     = ["o"]

    fig, ax = plt.subplots(figsize=(7.0, 4.2))

    x = df["epoch"].to_numpy()
    lines = []
    labels = []

    for (col, label), c, mk in zip(metrics, colors, markers):
        if col not in df.columns:
            continue
        y = df[col].to_numpy(dtype=float)

        # plot line
        (ln,) = ax.plot(
            x, y, lw=2.0, color=c, marker=mk, markevery=max(len(x)//12, 1),
            mec="white", mew=0.8, ms=5, alpha=0.95
        )
        lines.append(ln)
        labels.append(label)

    # axes formatting
    buffer = 1
    ax.set_xlim(x.min()-buffer, x.max()+buffer)
    ax.set_ylim(0, 0.1)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    # ax.set_title("Loss on the evaluation set over training")

    # minor ticks/grid for readability
    ax.minorticks_on()
    ax.grid(which="major", axis="both", linestyle="--", alpha=0.30)
    ax.grid(which="minor", axis="y", linestyle=":", alpha=0.20)

    # legend outside to avoid covering data
    ax.legend(lines, labels, title="Metric", frameon=True, framealpha=0.9,
              loc="upper right",)

    fig.tight_layout()
    plt.show()

# Usage:
plot_training_metrics(df)


In [ ]:
import math
import random
from pathlib import Path
from typing import Iterable, List, Tuple

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from automated_underwater_area_estimation.segmentation_quadrant.model import QuadrantSegmentationModel


def _load_rgb(path: Path) -> np.ndarray:
    """Load an image as RGB uint8 array (H, W, 3)."""
    return np.asarray(Image.open(path).convert("RGB"))


def _overlay_mask(img_rgb: np.ndarray, mask_bool: np.ndarray,
                  color: Tuple[int, int, int] = (0, 0, 255),
                  alpha: float = 0.35) -> np.ndarray:
    """
    Alpha-blend a solid color on top of img where mask is True.
    img_rgb: (H,W,3) uint8, mask_bool: (H,W) bool.
    """
    out = img_rgb.astype(np.float32)
    c = np.array(color, dtype=np.float32)[None, None, :]  # (1,1,3)
    m = mask_bool.astype(np.float32)[..., None]           # (H,W,1) 0/1
    out = out * (1.0 - alpha * m) + c * (alpha * m)
    return np.clip(out, 0, 255).astype(np.uint8)


def segment_folder_grid(
    folder: str | Path,
    n_images: int = 8,
    *,
    seed: int | None = None,
    cols: int = 4,
    alpha: float = 0.8,
    overlay_color: Tuple[int, int, int] = (128, 0, 128),  # blue in RGB
    figsize_per_cell: Tuple[float, float] = (4.0, 3.0),
    image_exts: Iterable[str] = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"),
) -> List[Path]:
    """
    Randomly sample images from `folder`, segment each, overlay the mask,
    and display results in a grid.

    Returns:
        List[Path]: the paths of the sampled images (in display order).
    """
    folder = Path(folder)
    exts = {e.lower() for e in image_exts}
    all_imgs = [p for p in folder.iterdir() if p.suffix.lower() in exts]
    if not all_imgs:
        raise FileNotFoundError(f"No images with extensions {sorted(exts)} found in: {folder}")

    if seed is not None:
        random.seed(seed)

    sample = random.sample(all_imgs, k=min(n_images, len(all_imgs)))

    # init model once
    model = QuadrantSegmentationModel()

    # prepare grid size
    cols = max(1, cols)
    rows = math.ceil(len(sample) / cols)
    fig_w = max(1, int(cols * figsize_per_cell[0]))
    fig_h = max(1, int(rows * figsize_per_cell[1]))

    plt.figure(figsize=(fig_w, fig_h), dpi=150)
    for i, img_path in enumerate(sample, start=1):
        img = _load_rgb(img_path)
        # model returns mask; ensure boolean HxW
        mask = model.segment_image(Image.fromarray(img))
        if not isinstance(mask, np.ndarray):
            try:
                mask = mask.detach().cpu().numpy()
            except Exception:
                mask = np.asarray(mask)
        if mask.ndim == 3:
            # reduce any (C,H,W) or (H,W,1) to (H,W)
            mask = mask.max(axis=0) if mask.shape[0] in (1, 3) else mask[..., 0]
        mask = (mask > 0.5)

        overlay = _overlay_mask(img, mask, color=overlay_color, alpha=alpha)

        ax = plt.subplot(rows, cols, i)
        ax.imshow(overlay)
        ax.set_title("")         # no per-image title
        ax.axis("off")

    plt.tight_layout(pad=0.3)
    plt.show()

    return sample


In [ ]:
# Show a 3×3 grid of 9 random images from a folder
_ = segment_folder_grid(
    folder="./automated_underwater_area_estimation/data_preprocessed/IBF/images",
    n_images=15,
    cols=4,
    alpha=0.6,
    figsize_per_cell=(4, 3),
    overlay_color=(255, 165 ,0),
    seed=42,
)


# Area estimation

In [ ]:
import pandas as pd

df = pd.read_csv("./automated_underwater_area_estimation/area_estimation/quadrant_predictions.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-colorblind')

# compute relative error if not given
df["rel_error"] = np.abs(df["pred_area_cm2"] - df["gt_area_cm2"]) / df["gt_area_cm2"]

# flag within 10%
threshold = 0.10
df["within10pct"] = df["rel_error"] <= threshold

# summary
n = len(df)
n_within = df["within10pct"].sum()

# scatter plot
fig1 = plt.figure(figsize=(10,8))
plt.scatter(df["gt_area_cm2"], df["pred_area_cm2"],
            c=df["within10pct"].map({True:"green", False:"red"}),
            alpha=0.7,
            s=100)
minv = min(df["gt_area_cm2"].min(), df["pred_area_cm2"].min())
maxv = max(df["gt_area_cm2"].max(), df["pred_area_cm2"].max())
plt.plot([minv, maxv], [minv, maxv], 'k--', label="Perfect prediction")
plt.plot([minv, maxv], [minv*(1+threshold), maxv*(1+threshold)], 'b--', label=f"+{threshold*100:.0f}% band")
plt.plot([minv, maxv], [minv*(1-threshold), maxv*(1-threshold)], 'b--', label=f"-{threshold*100:.0f}% band")
plt.xlabel("Ground truth PAE", fontsize=20)
plt.ylabel("Estimated PAE", fontsize=20)
# plt.title("Predicted vs Ground Truth", fontsize=26)
plt.legend(fontsize=20)
plt.grid(True)
plt.xlim(0.00025, 0.0011)
plt.ylim(0.00025, 0.0011)
plt.tight_layout()
plt.tick_params(axis='both', which='major', labelsize=16)

plt.show()
plt.close(fig1)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.style.use('seaborn-v0_8-colorblind')

# keep values as fractions in [0,1], just *display* as percentages
rel = df["rel_error"].astype(float).dropna().to_numpy()
threshold = 0.10  # 10%

fig2, ax = plt.subplots(figsize=(5, 8), dpi=300)

ax.boxplot(
    rel,
    vert=True,
    showmeans=True,
    meanline=True,
    patch_artist=True,
    boxprops=dict(alpha=0.9),
    whiskerprops=dict(alpha=0.9),
    capprops=dict(alpha=0.9),
    medianprops=dict(linewidth=2),
    meanprops=dict(color='black', linewidth=1.5),
)

# Cosmetic: single x tick label
ax.set_xticks([1])
ax.set_xticklabels(["Relative error"], fontsize=18)

# Y axis range (robust to outliers)
y_max = float(np.percentile(rel, 99.5)) * 1.1 if rel.size else 1.0
y_max = max(y_max, rel.max() * 1.05 if rel.size else 1.0)
ax.set_ylim(0, min(1.0, y_max))  # clamp to 100%

# Threshold reference line at 10%
ax.axhline(threshold, linestyle="--", color="#1f77b4", alpha=0.8, label="10% threshold")

# Format y-axis as percentages
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))

ax.set_ylabel("Relative error (%)", fontsize=18)
ax.set_title("Relative Error Distribution", fontsize=18, pad=8)
ax.grid(axis="y", linestyle=":", alpha=0.5)
# ax.legend(loc="upper right", frameon=True, framealpha=0.95, fontsize=14)

ax.tick_params(axis='y', labelsize=14)
plt.tight_layout()
plt.show()
plt.close(fig2)


In [ ]:
import math
from pathlib import Path
from typing import List, Tuple, Optional, Iterable, Union, Dict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Polygon

# -------------------------
# Prediction helpers (your logic)
# -------------------------
def to_mask2d(m_or_path: Union[str, np.ndarray]) -> np.ndarray:
    m = np.asarray(m_or_path)
    m = (np.squeeze(m) > 0).astype(np.uint8)
    if m.ndim != 2:
        raise ValueError(f"Expected 2D mask, got shape {m.shape}")
    return m

def point_bottom_right(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    if ys.size == 0: raise ValueError("Mask is empty.")
    i = (ys.astype(np.int64) * xs.astype(np.int64)).argmax()
    return int(ys[i]), int(xs[i])

def point_top_left(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    keep = (ys > 0) & (xs > 0)
    ys, xs = ys[keep], xs[keep]
    if ys.size == 0: raise ValueError("No white pixels with x>0 and y>0.")
    sums = ys.astype(np.int64) + xs.astype(np.int64)
    min_sum = sums.min()
    idx = np.where(sums == min_sum)[0]
    y_sel = int(ys[idx].min())
    x_sel = int(xs[idx][ys[idx] == y_sel].min())
    return y_sel, x_sel

def point_top_right(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    diffs = xs.astype(np.int64) - ys.astype(np.int64)
    best = diffs.max()
    keep = (diffs == best)
    xs_t, ys_t = xs[keep], ys[keep]
    x_sel = int(xs_t.max())
    y_sel = int(ys_t[xs_t == x_sel].min())
    return y_sel, x_sel

def point_bottom_left(m: np.ndarray) -> Tuple[int, int]:
    ys, xs = np.where(m == 1)
    diffs = ys.astype(np.int64) - xs.astype(np.int64)
    best = diffs.max()
    keep = (diffs == best)
    ys_t, xs_t = ys[keep], xs[keep]
    y_sel = int(ys_t.max())
    x_sel = int(xs_t[ys_t == y_sel].min())
    return y_sel, x_sel

def find_four_points(mask_2d: Union[str, np.ndarray]) -> Dict[str, Tuple[int,int]]:
    m = to_mask2d(mask_2d)
    return {
        "TL": point_top_left(m),
        "TR": point_top_right(m),
        "BR": point_bottom_right(m),
        "BL": point_bottom_left(m),
    }

def overlay_color_mask(img_rgb: np.ndarray, mask_bool: np.ndarray,
                       color=(0, 0, 255), alpha: float = 0.35) -> np.ndarray:
    out = img_rgb.astype(np.float32)
    c = np.array(color, dtype=np.float32)[None, None, :]
    m = mask_bool.astype(np.float32)[..., None]
    out = out * (1.0 - alpha * m) + c * (alpha * m)
    return np.clip(out, 0, 255).astype(np.uint8)

# -------------------------
# Ground-truth (CPC) helpers (your logic)
# -------------------------
import re
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}
FLOAT = r"[+-]?(?:\d+(?:\.\d+)?|\.\d+)"
PAIR_RE = re.compile(rf'^\s*"?\s*({FLOAT})\s*"?\s*,\s*"?\s*({FLOAT})\s*"?\s*$')
NUMS_RE = re.compile(rf"{FLOAT}")

def _read_text(path: Path) -> str:
    for enc in ("utf-8-sig", "latin-1"):
        try: return path.read_text(encoding=enc, errors="strict")
        except Exception: pass
    return path.read_text(encoding="utf-8", errors="ignore")

def parse_cpc(path: Path):
    text = _read_text(path)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip() != ""]
    if not lines:
        raise ValueError(f"Empty CPC: {path}")
    nums = [float(s) for s in NUMS_RE.findall(lines[0])]
    if len(nums) < 4:
        raise ValueError(f"Unexpected CPC header format: {path}")
    work_w, work_h = float(nums[-4]), float(nums[-3])
    roi = []
    i = 1
    while i < len(lines) and len(roi) < 4:
        m = PAIR_RE.match(lines[i])
        if m:
            roi.append((float(m.group(1)), float(m.group(2))))
        i += 1
    if len(roi) != 4:
        raise ValueError(f"Could not find 4 ROI corners in {path}")
    return work_w, work_h, roi

def _scale_points_to_pixels(roi, work_w, work_h, img_w, img_h):
    sx, sy = img_w / work_w, img_h / work_h
    return [(x * sx, y * sy) for (x, y) in roi]

# -------------------------
# Main function
# -------------------------
from automated_underwater_area_estimation.segmentation_quadrant.model import QuadrantSegmentationModel

def juxtapose_prediction_vs_gt(
    image_paths: List[Path | str],
    cpcs_dir: Path | str,
    *,
    model: Optional[QuadrantSegmentationModel] = None,
    alpha: float = 0.35,
    overlay_color=(0, 0, 255),               # prediction overlay color (RGB)
    pt_color="#FFBF00", side_color="#FFBF00", diag_color="#FFBF00",
    figsize_per_row: Tuple[float, float] = (12.0, 4.0),  # width × height per row (2 columns total)
    titlefont: int = 16,
    show_titles: bool = True,
):
    """
    For each image in `image_paths`, draws two panels in a single figure row:
      Left : Prediction overlay with quadrant corner points + lines
      Right: Ground-truth overlay from matching .cpc file

    Assumes a GT CPC file named <stem>.cpc lives in `cpcs_dir`.

    Returns:
        (fig, axes): Matplotlib figure and axes array of shape (N, 2)
    """
    paths = [Path(p) for p in image_paths]
    cpcs_dir = Path(cpcs_dir)
    if model is None:
        model = QuadrantSegmentationModel()

    n = len(paths)
    if n == 0:
        raise ValueError("No image paths provided.")
    fig, axes = plt.subplots(
        nrows=n, ncols=2, figsize=(figsize_per_row[0], figsize_per_row[1] * n),
        dpi=150, squeeze=False
    )

    for row, img_path in enumerate(paths):
        stem = img_path.stem
        cpc_path = cpcs_dir / f"{stem}.cpc"
        if not img_path.exists():
            raise FileNotFoundError(f"Image not found: {img_path}")
        if not cpc_path.exists():
            raise FileNotFoundError(f"CPC not found: {cpc_path}")

        # --- load image ---
        img = np.asarray(Image.open(img_path).convert("RGB"))

        # --- PREDICTION PANEL (left) ---
        axL = axes[row, 0]
        # predict mask
        pred = model.segment_image(Image.fromarray(img))
        if not isinstance(pred, np.ndarray):
            try: pred = pred.detach().cpu().numpy()
            except Exception: pred = np.asarray(pred)
        if pred.ndim == 3:  # (C,H,W) or (H,W,1)
            pred = pred.max(axis=0) if pred.shape[0] in (1, 3) else pred[..., 0]
        mask_bool = (pred > 0.5)

        overlay_pred = overlay_color_mask(img, mask_bool, color=overlay_color, alpha=alpha)
        axL.imshow(overlay_pred)

        # find 4 extreme points & draw lines
        m2d = to_mask2d(mask_bool.astype(np.uint8))
        pts = find_four_points(m2d)
        # points
        for k, (y, x) in pts.items():
            axL.scatter([x], [y], s=150, facecolor="white", edgecolor="black", zorder=4)
            axL.scatter([x], [y], s=70, color=pt_color, zorder=5)
        # sides and diagonals
        sides = [("TR", "TL"), ("TR", "BR"), ("BL", "BR"), ("TL", "BL")]
        diags = [("TL", "BR"), ("TR", "BL")]
        def _seg(a, b, color, lw=2.2, z=2):
            (y1, x1), (y2, x2) = pts[a], pts[b]
            axL.plot([x1, x2], [y1, y2], color=color, lw=lw, alpha=0.9, zorder=z)
        for a, b in sides: _seg(a, b, side_color, 2.2, 2)
        for a, b in diags: _seg(a, b, diag_color, 2.4, 1)

        if show_titles:
            axL.set_title("Prediction", fontsize=titlefont)
        axL.axis("off")

        # --- GROUND-TRUTH PANEL (right) ---
        axR = axes[row, 1]
        work_w, work_h, roi = parse_cpc(cpc_path)
        h, w = img.shape[0], img.shape[1]
        roi_px = _scale_points_to_pixels(roi, work_w, work_h, w, h)

        axR.imshow(img)
        poly = Polygon(np.array(roi_px, dtype=np.float32), closed=True, fill=False,
                       linewidth=4, color="#d95f02")
        axR.add_patch(poly)
        xs, ys = zip(*roi_px)
        axR.scatter(xs, ys, s=70, color="#1b9e77")
        if show_titles:
            axR.set_title("Ground truth", fontsize=titlefont)
        axR.axis("off")

    plt.tight_layout(pad=0.2)
    return fig, axes


In [ ]:
# List of images you want to compare (any order)
images = df.query("rel_error > 0.1")["image_path"].tolist()
# Folder that contains the matching .cpc files (same stem names)
cpcs_dir = "./automated_underwater_area_estimation/data_preprocessed/IBF/cpcs"

fig, axes = juxtapose_prediction_vs_gt(
    images, cpcs_dir,
    alpha=0.35,
    overlay_color=(0,0,255),   # blue overlay for prediction
    titlefont=16,
    show_titles=True
)
plt.show()
